In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**1. Ładowanie dokumentów**

Najpierw pobrałam z https://ui.adsabs.harvard.edu 5 najbardziej interesujące mnie artykuły, 2 z których to są wywiady z dwoma laureatami Nagrody Nobla. Pozostałe artykuły opisują nowatorskie podejścia w optycznych i kwantowych technologiach

Ładowanie pdfów do folderu

In [ ]:
from google.colab import files
import os


In [ ]:
folder_path = '/content/pdfs'
os.makedirs(folder_path, exist_ok=True)
articles = files.upload()

for article in articles.keys():
  os.rename(article, os.path.join('/content/pdfs', article))

Saving article5.pdf to article5.pdf


In [ ]:
articles_in_folder = os.listdir(folder_path)
print("Articles in the folder:")
for article in articles_in_folder:
  print(article)

Articles in the folder:
article2.pdf
article1.pdf
article4.pdf
article5.pdf
article3.pdf


Wszystkie artykuły są juz w floderu na dysku

**2. Wyodrębnianie tekstu z pdfów i podział tekstu na fragmenty**

instalacja bibliotek

In [ ]:
!pip install langchain sentence-transformers faiss-cpu pypdf transformers torch langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

all_articles_in_one = []

for article in articles_in_folder:
  loader = PyPDFLoader(os.path.join(folder_path, article))
  pages = loader.load()
  all_articles_in_one.extend(pages)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 200,
    length_function = len,
    separators=["\n\n", "\n", " ", ""]
)
spleted_articles = text_splitter.split_documents(all_articles_in_one)

Wyświtlienie liczby fragmentów

In [ ]:
print("Number of splited articles: ", len(spleted_articles))

Number of splited articles:  797


**3. Tworzenie wektorowej bd**

Jako model wybrałam *intfloat/multilingual-e5-large*, bo ten model jest w miare prosty i idealnie pasuje do artykułów w języku angelskim

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
wektorowa_baza_danych = FAISS.from_documents(spleted_articles, embeddings)
retriever = wektorowa_baza_danych.as_retriever()

/tmp/ipython-input-54156045.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

**4. Inicjalizacja modelu dla odpowiedzi**

Jako model wybrałam google/flan-t5-base, w notatniku był użyty google/flan-t5-small, więc postanowiłam, że mogę użyć google/flan-t5-base. Najpierw sprawdzałam jak pracuje t5-base, ale wyniki byli fatalne, w odpowiedzi zamiast tekstu była wartość True, więc wróciłam do google/flan-t5-base

In [33]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain.llms import HuggingFacePipeline

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

pipline = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=512)
llm = HuggingFacePipeline(pipeline=pipline)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Device set to use cpu


**5. Tworzenie łańcuchów rag z pamięcią**

In [37]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True, output_key='answer')
conversation = ConversationalRetrievalChain.from_llm(llm=llm,retriever=retriever,memory=memory,return_source_documents=True)

**6. Cytowanie bez halucynacji**

In [38]:
def cytowanie(question: str):
  result = conversation({'question': question})
  if not result.get('source_documents'):
    return 'The answer to this question has not been found'
  else:
    answer_text = result.get('answer', '')
    source_info = 'Sources:\n'
    for doc in result['source_documents']:
      source_info += f'Source: {doc.metadata.get("source", "Unknown source")}\n'
    return f'{answer_text}\n\n{source_info}'

**7. Conversation**

In [39]:
import sys

print('If you want to stop the conversation, plese, enter (exit)')
while True:
  question = input('Your cuestions: ')
  if question.lower() == 'exit':
    break
  answer = cytowanie(question)
  print('Answer: ')
  print(answer)


If you want to stop the conversation, plese, enter (exit)
Your cuestions: What are the key advantages of using optical computing platforms for next-generation machine learning hardware?
Answer: 
computing speed, energy efficiency, and parallelism

Sources:
Source: /content/pdfs/article3.pdf
Source: /content/pdfs/article3.pdf
Source: /content/pdfs/article3.pdf
Source: /content/pdfs/article4.pdf

Your cuestions: How can parallel optical computing increase computational power without increasing the size of a photonic chip?


Token indices sequence length is longer than the specified maximum sequence length for this model (529 > 512). Running this sequence through the model will result in indexing errors


Answer: 
on-chip cycles can be determined, and a chip-reuse strat- egy can be employed to efficiently process long sequence signals

Sources:
Source: /content/pdfs/article4.pdf
Source: /content/pdfs/article4.pdf
Source: /content/pdfs/article5.pdf
Source: /content/pdfs/article4.pdf

Your cuestions: What is the Kosterlitz-Thouless transition and what role do vortex-antivortex pairs play in it?
Answer: 
Vortex-antivortex pairs play a key role in phase tran - sitions, leading to a topological phase transition now known as the Kosterlitz-Thouless (KT) transition.

Sources:
Source: /content/pdfs/article2.pdf
Source: /content/pdfs/article2.pdf
Source: /content/pdfs/article2.pdf
Source: /content/pdfs/article2.pdf

Your cuestions: What is the population of Paris?
Answer: 
No

Sources:
Source: /content/pdfs/article1.pdf
Source: /content/pdfs/article3.pdf
Source: /content/pdfs/article3.pdf
Source: /content/pdfs/article3.pdf

Your cuestions: What is the ideal temperature for growing tomatoes?
Answ